In [ ]:
import os, json, random
import pandas as pd
import numpy as np
from itertools import combinations

# ============================================================
# Benchmark generator MUS-2(ANY Repair)
# ============================================================

# ----------------------------
# Config
# ----------------------------
TARGET_QUERIES = 41
SEED = 11
random.seed(SEED)
rng = np.random.default_rng(SEED)

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

BASE_MIN = 40
BASE_MAX = 220

CAT_EXTRA_COLS = ["Model", "Vehicle Size", "Engine Fuel Type"]
NUM_EXTRA_SPECS = [
    ("Year", ">=", [0.6, 0.7, 0.8, 0.9]),
    ("city mpg", ">=", [0.6, 0.7, 0.8, 0.9]),
    ("highway MPG", ">=", [0.6, 0.7, 0.8, 0.9]),
    ("MSRP", "<=", [0.1, 0.2, 0.3, 0.4]),
]

MSRP_MIN_CAP = 12000
MIN_CAT_SUPPORT_IN_U = 3

MAX_PREDS_PER_COL = 25
MAX_TOTAL_PREDS = 70

OUT_PATH = "./benchmark_queries_10_k2.json"

# ----------------------------
# Load data
# ----------------------------
path_candidates = ["./data.csv", "/mnt/data/data.csv"]
path = next((p for p in path_candidates if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

df0 = pd.read_csv(path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Bitset helpers
# ----------------------------
def make_bitset_from_bool(mask_bool: np.ndarray) -> int:
    bits = 0
    for i, v in enumerate(mask_bool):
        if v:
            bits |= (1 << i)
    return bits

def popcount(x: int) -> int:
    return x.bit_count()

# ----------------------------
# Candidate base groups
# ----------------------------
group = df.groupby(BASE_COLS).size().reset_index(name="count")
eligible_bases = group[(group["count"] >= BASE_MIN) & (group["count"] <= BASE_MAX)].to_dict("records")
random.shuffle(eligible_bases)

def get_base_indices(base_rec) -> np.ndarray:
    mask = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        mask &= (df[c].values == base_rec[c])
    return np.where(mask)[0]

# ----------------------------
# Build candidate additional predicates inside U as bitsets over U
# ----------------------------
def build_predicates_in_U(U: pd.DataFrame):
    preds = []

    # categorical
    for col in CAT_EXTRA_COLS:
        if col not in U.columns:
            continue
        vc = U[col].value_counts()
        values = vc[vc >= MIN_CAT_SUPPORT_IN_U].index.tolist()[:MAX_PREDS_PER_COL]
        for v in values:
            maskU = (U[col].values == v)
            if maskU.all() or (~maskU).all():
                continue
            preds.append({"col": col, "op": "==", "value": v, "bits": make_bitset_from_bool(maskU)})

    # numeric from U quantiles
    for col, op, qs in NUM_EXTRA_SPECS:
        if col not in U.columns:
            continue
        quant_vals = U[col].quantile(qs).values
        thresholds = sorted(set(int(round(x)) for x in quant_vals))[:MAX_PREDS_PER_COL]
        for t in thresholds:
            if col == "MSRP" and t < MSRP_MIN_CAP:
                continue
            maskU = (U[col].values >= t) if op == ">=" else (U[col].values <= t)
            if maskU.all() or (~maskU).all():
                continue
            preds.append({"col": col, "op": op, "value": int(t), "bits": make_bitset_from_bool(maskU)})

    # dedup
    uniq = {}
    for p in preds:
        uniq[(p["col"], p["op"], p["value"])] = p
    preds = list(uniq.values())

    # prune too-weak/too-strong
    nU = len(U)
    filtered = []
    for p in preds:
        cnt = popcount(p["bits"])
        if cnt < 2 or cnt > int(0.9 * nU):
            continue
        filtered.append(p)
    preds = filtered

    # cap total preds (prefer informative)
    by_col = {}
    for p in preds:
        by_col.setdefault(p["col"], []).append(p)
    for col in by_col:
        by_col[col].sort(key=lambda p: abs(popcount(p["bits"]) - nU/2))

    capped = []
    for col, plist in by_col.items():
        capped.extend(plist[:MAX_PREDS_PER_COL])
    capped.sort(key=lambda p: abs(popcount(p["bits"]) - nU/2))
    return capped[:MAX_TOTAL_PREDS]

# ----------------------------
# Find 2 constraints {a,b} such that:
#   a & b == 0   (empty within U)
#   a != 0 and b != 0 (drop-any-one => non-empty)
# Require distinct columns if requested.
# ----------------------------
def find_drop_any_one_pair(preds, require_distinct_cols=True, max_tries=20000):
    if len(preds) < 2:
        return None

    # shuffle indices to diversify
    idxs = list(range(len(preds)))
    random.shuffle(idxs)

    # try random pairs first (fast)
    for _ in range(max_tries):
        i, j = random.sample(idxs, 2)
        if require_distinct_cols and preds[i]["col"] == preds[j]["col"]:
            continue

        bi = preds[i]["bits"]
        bj = preds[j]["bits"]

        # need each individually non-empty
        if bi == 0 or bj == 0:
            continue

        # full empty
        if (bi & bj) != 0:
            continue

        return [preds[i], preds[j]]

    # fallback exhaustive if random misses (still manageable at MAX_TOTAL_PREDS=70)
    n = len(preds)
    for i in range(n):
        for j in range(i+1, n):
            if require_distinct_cols and preds[i]["col"] == preds[j]["col"]:
                continue
            bi, bj = preds[i]["bits"], preds[j]["bits"]
            if bi != 0 and bj != 0 and (bi & bj) == 0:
                return [preds[i], preds[j]]

    return None

# ----------------------------
# Natural language rendering
# ----------------------------
def clause(col, op, value):
    if op == "==":
        if col == "Engine Fuel Type":
            return f"runs on {value}"
        if col == "Vehicle Size":
            return f"is {str(value).lower()} size"
        if col == "Model":
            return f"is a {value}"
        return f"has {col} {value}"

    v = int(value)
    if col == "Year" and op == ">=":
        return f"is from {v} or newer"
    if col == "city mpg" and op == ">=":
        return f"gets at least {v} city MPG"
    if col == "highway MPG" and op == ">=":
        return f"gets at least {v} highway MPG"
    if col == "MSRP" and op == "<=":
        return f"has MSRP at most ${v:,}"
    return f"has {col} {op} {v}"

def make_base_query_sentence(base_rec):
    return (
        f"I am looking for a {base_rec['Make']} {base_rec['Vehicle Style']} "
        f"with a {base_rec['Transmission Type']} transmission and {base_rec['Driven_Wheels']}."
    )

def make_full_query_sentence(base_rec, additional):
    base_part = (
        f"I am looking for a {base_rec['Make']} {base_rec['Vehicle Style']} "
        f"with a {base_rec['Transmission Type']} transmission and {base_rec['Driven_Wheels']}"
    )
    extras = ", ".join(clause(p["col"], p["op"], p["value"]) for p in additional)
    return base_part + ", " + extras + "."

# ----------------------------
# Weights + recommendation logic (k=2)
# ----------------------------
def sample_weights(additional, rng: np.random.Generator):
    raw = rng.uniform(0.2, 1.0, size=len(additional))
    w = raw / raw.sum()
    out = []
    for p, wi in zip(additional, w):
        out.append({"column": p["col"], "op": p["op"], "value": p["value"], "weight": float(wi)})
    return out

def apply_constraint_to_df(df_in: pd.DataFrame, p: dict) -> np.ndarray:
    col, op, value = p["col"], p["op"], p["value"]
    if op == "==":
        return (df_in[col].values == value)
    if op == ">=":
        return (df_in[col].values >= int(value))
    if op == "<=":
        return (df_in[col].values <= int(value))
    raise ValueError(f"Unsupported op: {op}")

def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x)); xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def recommend_car_for_instance(U: pd.DataFrame, additional: list, weighted_constraints: list):
    """
    For k=2:
      - Full query is infeasible by construction (U∩a∩b empty).
      - Relax exactly one: lowest weight.
      - Recommend among candidates satisfying the remaining one constraint,
        using soft weighted score over BOTH constraints.
    """
    w_sorted = sorted(weighted_constraints, key=lambda d: d["weight"])
    relaxed = w_sorted[0]  # lowest weight constraint to relax

    keep = [p for p in additional if not (p["col"] == relaxed["column"] and p["op"] == relaxed["op"] and p["value"] == relaxed["value"])]

    mask = np.ones(len(U), dtype=bool)
    for p in keep:
        mask &= apply_constraint_to_df(U, p)

    candidates = U[mask].copy()
    if len(candidates) == 0:
        return {"relaxed_constraint": relaxed, "recommended_car": None}

    # soft scoring over BOTH constraints
    scores = np.zeros(len(candidates), dtype=float)
    norms = {}
    for col in ["Year", "city mpg", "highway MPG", "MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)
    cand_pos_in_U = candidates.index.values

    for wc in weighted_constraints:
        col, op, val, w = wc["column"], wc["op"], wc["value"], wc["weight"]
        if op == "==":
            scores += w * (candidates[col].values == val).astype(float)
        elif op == ">=":
            scores += w * (norms[col][cand_pos_in_U] if col in norms else (candidates[col].values >= int(val)).astype(float))
        elif op == "<=":
            scores += w * ((1.0 - norms[col][cand_pos_in_U]) if col in norms else (candidates[col].values <= int(val)).astype(float))

    msrp = candidates["MSRP"].values.astype(float) if "MSRP" in candidates.columns else np.zeros(len(candidates), dtype=float)
    best_idx = np.lexsort((msrp, -scores))[0]
    best_row = candidates.iloc[best_idx]

    recommended = {
        "Make": best_row.get("Make", None),
        "Model": best_row.get("Model", None),
        "Year": int(best_row["Year"]) if "Year" in best_row else None,
        "Engine Fuel Type": best_row.get("Engine Fuel Type", None),
        "Transmission Type": best_row.get("Transmission Type", None),
        "Driven_Wheels": best_row.get("Driven_Wheels", None),
        "Vehicle Size": best_row.get("Vehicle Size", None),
        "Vehicle Style": best_row.get("Vehicle Style", None),
        "city mpg": int(best_row["city mpg"]) if "city mpg" in best_row else None,
        "highway MPG": int(best_row["highway MPG"]) if "highway MPG" in best_row else None,
        "MSRP": int(best_row["MSRP"]) if "MSRP" in best_row else None,
    }
    return {"relaxed_constraint": relaxed, "recommended_car": recommended}

# ----------------------------
# Generate TARGET_QUERIES and dump JSON
# ----------------------------
results = []
seen = set()

for base_rec in eligible_bases:
    if len(results) >= TARGET_QUERIES:
        break

    base_sig = tuple(base_rec[c] for c in BASE_COLS)
    if base_sig in seen:
        continue

    U_idx = get_base_indices(base_rec)
    if not (BASE_MIN <= len(U_idx) <= BASE_MAX):
        continue

    U = df.iloc[U_idx].reset_index(drop=True)

    preds = build_predicates_in_U(U)
    additional = find_drop_any_one_pair(preds, require_distinct_cols=True)
    if additional is None:
        continue

    seen.add(base_sig)

    base_query_sentence = make_base_query_sentence(base_rec)
    full_query_sentence = make_full_query_sentence(base_rec, additional)

    additional_constraints = [{"column": p["col"], "op": p["op"], "value": p["value"]} for p in additional]
    weighted_constraints = sample_weights(additional, rng)

    rec = recommend_car_for_instance(
        U=U,
        additional=additional,
        weighted_constraints=weighted_constraints
    )

    results.append({
        "base_query_sentence": base_query_sentence,
        "full_query_sentence": full_query_sentence,
        "additional_constraints": additional_constraints,   # 2 constraints
        "constraint_weights": weighted_constraints,         # 2 weights
        "relaxation_policy": "relax_lowest_weight_constraint",
        "chosen_relaxation": rec["relaxed_constraint"],
        "recommended_car": rec["recommended_car"]
    })

if len(results) < TARGET_QUERIES:
    raise RuntimeError(
        f"Only generated {len(results)} queries. "
        f"Try widening BASE_MAX, lowering require_distinct_cols, or increasing MAX_TOTAL_PREDS."
    )

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(results)} queries to: {OUT_PATH}")
print("Example record:\n", json.dumps(results[0], indent=2, ensure_ascii=False))


In [ ]:
import os, json, random
import pandas as pd
import numpy as np
from itertools import combinations

# ----------------------------
# MUS-4(UNIQUE Repair)
# ----------------------------
TARGET_QUERIES = 41
SEED = 11
random.seed(SEED)
rng = np.random.default_rng(SEED)

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

BASE_MIN = 40
BASE_MAX = 220

# Candidate additional-constraint columns
CAT_EXTRA_COLS = ["Model", "Vehicle Size", "Engine Fuel Type"]
NUM_EXTRA_SPECS = [
    ("Year", ">=", [0.6, 0.7, 0.8, 0.9]),
    ("city mpg", ">=", [0.6, 0.7, 0.8, 0.9]),
    ("highway MPG", ">=", [0.6, 0.7, 0.8, 0.9]),
    ("MSRP", "<=", [0.1, 0.2, 0.3, 0.4]),
]

MSRP_MIN_CAP = 12000
MIN_CAT_SUPPORT_IN_U = 3

# Keep search manageable per base
MAX_PREDS_PER_COL = 25
MAX_TOTAL_PREDS = 90

OUT_PATH = "./benchmark_queries_onefix_10.json"

# ----------------------------
# Load data
# ----------------------------
path_candidates = ["./data.csv", "/mnt/data/data.csv"]
path = next((p for p in path_candidates if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

df0 = pd.read_csv(path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Bitset helpers (bitset over rows of U, not over full df)
# ----------------------------
def make_bitset_from_bool(mask_bool: np.ndarray) -> int:
    bits = 0
    for i, v in enumerate(mask_bool):
        if v:
            bits |= (1 << i)
    return bits

def popcount(x: int) -> int:
    return x.bit_count()

def bitset_to_indices(bits: int) -> np.ndarray:
    """Return indices where bit is 1 (0..|U|-1)."""
    out = []
    i = 0
    while bits:
        if bits & 1:
            out.append(i)
        bits >>= 1
        i += 1
    return np.array(out, dtype=int)

# ----------------------------
# Candidate base groups
# ----------------------------
group = df.groupby(BASE_COLS).size().reset_index(name="count")
eligible_bases = group[(group["count"] >= BASE_MIN) & (group["count"] <= BASE_MAX)].to_dict("records")
random.shuffle(eligible_bases)

def get_base_indices(base_rec) -> np.ndarray:
    mask = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        mask &= (df[c].values == base_rec[c])
    return np.where(mask)[0]

# ----------------------------
# Build predicates inside U as bitsets over U
# ----------------------------
def build_predicates_in_U(U: pd.DataFrame):
    preds = []

    # Categorical
    for col in CAT_EXTRA_COLS:
        if col not in U.columns:
            continue
        vc = U[col].value_counts()
        values = vc[vc >= MIN_CAT_SUPPORT_IN_U].index.tolist()[:MAX_PREDS_PER_COL]
        for v in values:
            maskU = (U[col].values == v)
            if maskU.all() or (~maskU).all():
                continue
            preds.append({"col": col, "op": "==", "value": v, "bits": make_bitset_from_bool(maskU)})

    # Numeric (quantiles inside U)
    for col, op, qs in NUM_EXTRA_SPECS:
        if col not in U.columns:
            continue
        quant_vals = U[col].quantile(qs).values
        thresholds = sorted(set(int(round(x)) for x in quant_vals))[:MAX_PREDS_PER_COL]

        for t in thresholds:
            if col == "MSRP" and t < MSRP_MIN_CAP:
                continue
            if op == ">=":
                maskU = (U[col].values >= t)
            else:
                maskU = (U[col].values <= t)
            if maskU.all() or (~maskU).all():
                continue
            preds.append({"col": col, "op": op, "value": int(t), "bits": make_bitset_from_bool(maskU)})

    # Dedup
    uniq = {}
    for p in preds:
        uniq[(p["col"], p["op"], p["value"])] = p
    preds = list(uniq.values())

    # Prune too-weak/too-strong predicates within U
    nU = len(U)
    filtered = []
    for p in preds:
        cnt = popcount(p["bits"])
        if cnt < 2 or cnt > int(0.9 * nU):
            continue
        filtered.append(p)
    preds = filtered

    # Cap total predicates (keep diverse and informative)
    by_col = {}
    for p in preds:
        by_col.setdefault(p["col"], []).append(p)

    for col in by_col:
        by_col[col].sort(key=lambda p: abs(popcount(p["bits"]) - nU / 2))

    capped = []
    for col, plist in by_col.items():
        capped.extend(plist[:MAX_PREDS_PER_COL])

    capped.sort(key=lambda p: abs(popcount(p["bits"]) - nU / 2))
    return capped[:MAX_TOTAL_PREDS]

# ----------------------------
# Find "one-fix" infeasible quad:
# Exactly ONE of the four constraints, when dropped, makes the set non-empty.
#
# Condition using triples:
# Pick a triple (i,j,k) that is NON-empty  -> this corresponds to dropping l
# Find l such that:
#   (i&j&k&l) is empty
#   and the other three triples that include l are empty:
#     (i&j&l)==0, (i&k&l)==0, (j&k&l)==0
#
# Then l is the UNIQUE repair constraint (drop l -> non-empty).
# ----------------------------
def find_one_fix_quad(preds, require_distinct_cols=True):
    if len(preds) < 4:
        return None

    n = len(preds)

    # List all non-empty triples
    triples = []
    for i, j, k in combinations(range(n), 3):
        if require_distinct_cols:
            cols = {preds[i]["col"], preds[j]["col"], preds[k]["col"]}
            if len(cols) < 3:
                continue
        bits_ijk = preds[i]["bits"] & preds[j]["bits"] & preds[k]["bits"]
        if bits_ijk != 0:
            triples.append((i, j, k, bits_ijk))

    if not triples:
        return None

    random.shuffle(triples)

    for i, j, k, bits_ijk in triples:
        for l in range(n):
            if l in (i, j, k):
                continue
            if require_distinct_cols:
                cols = {preds[i]["col"], preds[j]["col"], preds[k]["col"], preds[l]["col"]}
                if len(cols) < 4:
                    continue

            # Full intersection must be empty
            if (bits_ijk & preds[l]["bits"]) != 0:
                continue

            # All other 3-of-4 (triples including l) must be empty
            if (preds[i]["bits"] & preds[j]["bits"] & preds[l]["bits"]) != 0:
                continue
            if (preds[i]["bits"] & preds[k]["bits"] & preds[l]["bits"]) != 0:
                continue
            if (preds[j]["bits"] & preds[k]["bits"] & preds[l]["bits"]) != 0:
                continue

            quad = [preds[i], preds[j], preds[k], preds[l]]
            repair_idx = 3  # dropping preds[l] yields non-empty (bits_ijk)
            feasible_bits_after_repair = bits_ijk
            return quad, repair_idx, feasible_bits_after_repair

    return None

# ----------------------------
# Natural language rendering
# ----------------------------
def clause(col, op, value):
    if op == "==":
        if col == "Engine Fuel Type":
            return f"runs on {value}"
        if col == "Vehicle Size":
            return f"is {str(value).lower()} size"
        if col == "Model":
            return f"is a {value}"
        return f"has {col} {value}"

    v = int(value)
    if col == "Year" and op == ">=":
        return f"is from {v} or newer"
    if col == "city mpg" and op == ">=":
        return f"gets at least {v} city MPG"
    if col == "highway MPG" and op == ">=":
        return f"gets at least {v} highway MPG"
    if col == "MSRP" and op == "<=":
        return f"has MSRP at most ${v:,}"
    return f"has {col} {op} {v}"

def make_base_query_sentence(base_rec):
    return (
        f"I am looking for a {base_rec['Make']} {base_rec['Vehicle Style']} "
        f"with a {base_rec['Transmission Type']} transmission and {base_rec['Driven_Wheels']}."
    )

def make_full_query_sentence(base_rec, additional4):
    base_part = (
        f"I am looking for a {base_rec['Make']} {base_rec['Vehicle Style']} "
        f"with a {base_rec['Transmission Type']} transmission and {base_rec['Driven_Wheels']}"
    )
    extras = ", ".join(clause(p["col"], p["op"], p["value"]) for p in additional4)
    return base_part + ", " + extras + "."

# ----------------------------
# Weights: ensure the repair constraint is ALWAYS lowest
# ----------------------------
def sample_weights_one_fix(additional4, repair_idx, rng: np.random.Generator):
    """
    Assign weights (importance). The repair constraint is forced to be the smallest.
    """
    # High-ish weights for the other three
    hi = rng.uniform(0.35, 1.00, size=3)
    # Low weight for the repair constraint (so oracle relaxes it)
    lo = float(rng.uniform(0.01, 0.15))

    weights = []
    hi_iter = iter(hi.tolist())
    for i in range(4):
        if i == repair_idx:
            weights.append(lo)
        else:
            weights.append(float(next(hi_iter)))

    # Normalize to sum to 1
    w = np.array(weights, dtype=float)
    w = w / w.sum()

    out = []
    for p, wi in zip(additional4, w.tolist()):
        out.append({
            "column": p["col"],
            "op": p["op"],
            "value": p["value"],
            "weight": float(wi),
            "is_unique_repair": (p is additional4[repair_idx])
        })
    return out

# ----------------------------
# Recommendation (oracle)
# - relax the lowest-weight constraint (guaranteed to be the unique repair)
# - among candidates after relaxing it, choose best by weighted soft scoring
# ----------------------------
def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def recommend_car(U: pd.DataFrame, additional4, weighted_constraints, repair_idx):
    """
    Candidates are rows satisfying the 3 constraints other than the repair one.
    Then score candidates using all 4 constraints (repair has low weight).
    """
    # Hard-keep the other 3 constraints
    keep = [p for i, p in enumerate(additional4) if i != repair_idx]

    mask = np.ones(len(U), dtype=bool)
    for p in keep:
        col, op, val = p["col"], p["op"], p["value"]
        if op == "==":
            mask &= (U[col].values == val)
        elif op == ">=":
            mask &= (U[col].values >= int(val))
        elif op == "<=":
            mask &= (U[col].values <= int(val))
        else:
            raise ValueError(f"Unsupported op: {op}")

    candidates = U[mask].copy()
    if len(candidates) == 0:
        # Should not happen for one-fix instances, but keep safe.
        return None

    # Precompute normalization for numeric columns (within U)
    norms = {}
    for col in ["Year", "city mpg", "highway MPG", "MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)

    cand_pos_in_U = candidates.index.values
    scores = np.zeros(len(candidates), dtype=float)

    for wc in weighted_constraints:
        col, op, val, w = wc["column"], wc["op"], wc["value"], wc["weight"]

        if op == "==":
            sat = (candidates[col].values == val).astype(float)
            scores += w * sat

        elif op == ">=":
            # prefer higher actual values (smooth)
            if col in norms:
                scores += w * norms[col][cand_pos_in_U]
            else:
                sat = (candidates[col].values >= int(val)).astype(float)
                scores += w * sat

        elif op == "<=":
            # prefer lower actual values (smooth)
            if col in norms:
                scores += w * (1.0 - norms[col][cand_pos_in_U])
            else:
                sat = (candidates[col].values <= int(val)).astype(float)
                scores += w * sat

        else:
            raise ValueError(f"Unsupported op: {op}")

    # Tie-break: higher score, then lower MSRP if available
    if "MSRP" in candidates.columns:
        msrp = candidates["MSRP"].values.astype(float)
    else:
        msrp = np.zeros(len(candidates), dtype=float)

    best = np.lexsort((msrp, -scores))[0]
    row = candidates.iloc[best]

    return {
        "Make": row.get("Make", None),
        "Model": row.get("Model", None),
        "Year": int(row["Year"]) if "Year" in row else None,
        "Engine Fuel Type": row.get("Engine Fuel Type", None),
        "Transmission Type": row.get("Transmission Type", None),
        "Driven_Wheels": row.get("Driven_Wheels", None),
        "Vehicle Size": row.get("Vehicle Size", None),
        "Vehicle Style": row.get("Vehicle Style", None),
        "city mpg": int(row["city mpg"]) if "city mpg" in row else None,
        "highway MPG": int(row["highway MPG"]) if "highway MPG" in row else None,
        "MSRP": int(row["MSRP"]) if "MSRP" in row else None,
    }

# ----------------------------
# Generate TARGET_QUERIES and dump JSON
# ----------------------------
results = []
seen_bases = set()

for base_rec in eligible_bases:
    if len(results) >= TARGET_QUERIES:
        break

    base_sig = tuple(base_rec[c] for c in BASE_COLS)
    if base_sig in seen_bases:
        continue

    U_idx = get_base_indices(base_rec)
    if not (BASE_MIN <= len(U_idx) <= BASE_MAX):
        continue

    U = df.iloc[U_idx].reset_index(drop=True)

    preds = build_predicates_in_U(U)

    found = find_one_fix_quad(preds, require_distinct_cols=True)
    if found is None:
        continue

    additional4, repair_idx, feasible_bits = found
    # repair constraint is additional4[repair_idx] (by construction)

    # Confirm "one-fix" behavior (optional sanity; remove if you want)
    # - full empty:
    full_bits = additional4[0]["bits"] & additional4[1]["bits"] & additional4[2]["bits"] & additional4[3]["bits"]
    if full_bits != 0:
        continue
    # - only dropping repair is non-empty:
    #   dropping repair -> intersection of other 3
    bits_drop_repair = feasible_bits
    #   dropping any other -> must be empty
    #   (the constructor already enforced it, but we keep it as guard)
    def triple_bits(idxs):
        b = (1 << len(U)) - 1  # all-ones over U length is too huge if we do this; use AND of bits directly instead
        # We'll just do intersection directly:
        out = additional4[idxs[0]]["bits"] & additional4[idxs[1]]["bits"] & additional4[idxs[2]]["bits"]
        return out

    # Identify "other" drops:
    nonempty_triples = 0
    for drop_i in range(4):
        keep = [i for i in range(4) if i != drop_i]
        tb = additional4[keep[0]]["bits"] & additional4[keep[1]]["bits"] & additional4[keep[2]]["bits"]
        if tb != 0:
            nonempty_triples += 1
    if nonempty_triples != 1:
        continue

    # Build weights so repair is lowest
    weighted_constraints = sample_weights_one_fix(additional4, repair_idx, rng)

    # Oracle always relaxes the lowest-weight constraint (== repair)
    chosen_relaxation = {
        "column": additional4[repair_idx]["col"],
        "op": additional4[repair_idx]["op"],
        "value": additional4[repair_idx]["value"]
    }

    recommended = recommend_car(U, additional4, weighted_constraints, repair_idx)

    base_query_sentence = make_base_query_sentence(base_rec)
    full_query_sentence = make_full_query_sentence(base_rec, additional4)

    results.append({
        "base_query_sentence": base_query_sentence,
        "full_query_sentence": full_query_sentence,
        "additional_constraints": [
            {"column": p["col"], "op": p["op"], "value": p["value"]}
            for p in additional4
        ],
        "constraint_weights": weighted_constraints,
        "unique_repair_constraint": chosen_relaxation,
        "relaxation_policy": "relax_lowest_weight_constraint",
        "chosen_relaxation": chosen_relaxation,
        "recommended_car": recommended
    })

    seen_bases.add(base_sig)

if len(results) < TARGET_QUERIES:
    raise RuntimeError(
        f"Only generated {len(results)} one-fix queries. "
        f"Try widening BASE_MAX, lowering require_distinct_cols, or increasing MAX_TOTAL_PREDS."
    )

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(results)} queries to: {OUT_PATH}")
print("Example record:\n", json.dumps(results[0], indent=2, ensure_ascii=False))


In [ ]:
import os, json, random
import pandas as pd
import numpy as np
from typing import List
from itertools import combinations

# ============================================================
#  MUS-4(ANY Repair)
# ============================================================

# ----------------------------
# Config
# ----------------------------
TARGET_QUERIES = 40
SEED = 11
random.seed(SEED)
rng = np.random.default_rng(SEED)

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

# Keep base slices not too tiny, but allow bigger for higher MUS-4 hit rate
BASE_MIN = 35
BASE_MAX = 350  # quality is controlled by scoring below, not by forcing small U

CAT_EXTRA_COLS = ["Model", "Vehicle Size", "Engine Fuel Type"]
NUM_EXTRA_SPECS = [
    ("Year", ">="),
    ("city mpg", ">="),
    ("highway MPG", ">="),
    ("MSRP", "<="),
]

MSRP_MIN_CAP = 12000
MIN_CAT_SUPPORT_IN_U = 3

# More predicates increases success rate without lowering quality (runtime increases)
MAX_PREDS_PER_COL = 45
MAX_TOTAL_PREDS = 140

# How many bases to try (increase to get more queries)
MAX_BASE_TRIES = 4000

OUT_PATH = "./benchmark_queries_40_k4.json"

# ----------------------------
# Load data
# ----------------------------
path_candidates = ["./data.csv", "/mnt/data/data.csv"]
path = next((p for p in path_candidates if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

df0 = pd.read_csv(path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Bitset helpers
# ----------------------------
def make_bitset_from_bool(mask_bool: np.ndarray) -> int:
    bits = 0
    for i, v in enumerate(mask_bool):
        if v:
            bits |= (1 << i)
    return bits

def popcount(x: int) -> int:
    return x.bit_count()

# ----------------------------
# Candidate base groups
# ----------------------------
group = df.groupby(BASE_COLS).size().reset_index(name="count")
eligible_bases = group[(group["count"] >= BASE_MIN) & (group["count"] <= BASE_MAX)].to_dict("records")
random.shuffle(eligible_bases)

def get_base_indices(base_rec) -> np.ndarray:
    mask = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        mask &= (df[c].values == base_rec[c])
    return np.where(mask)[0]

# ----------------------------
# Build richer numeric thresholds from U (quality-preserving)
# ----------------------------
def numeric_thresholds(Ucol: pd.Series, colname: str, op: str) -> List[int]:
    x = Ucol.dropna().astype(float).values
    if len(x) == 0:
        return []
    xs = np.sort(np.unique(x))
    qs = np.quantile(xs, [0.60,0.70,0.80,0.90,0.95])
    q_int = sorted(set(int(round(v)) for v in qs))

    # Add near-extremes to enable UNSAT constructions
    # For >= constraints, include top few values.
    # For <= constraints (MSRP), include bottom few values.
    extra = []
    if op == ">=":
        extra_vals = xs[-min(6, len(xs)):]  # top
        extra.extend(int(round(v)) for v in extra_vals)
    else:  # "<="
        extra_vals = xs[:min(6, len(xs))]   # bottom
        extra.extend(int(round(v)) for v in extra_vals)

    # Also add a couple of "edge-ish" values (2nd/3rd from extreme)
    if len(xs) >= 10:
        if op == ">=":
            extra.extend(int(round(v)) for v in xs[-10:-6])
        else:
            extra.extend(int(round(v)) for v in xs[6:10])

    thr = sorted(set(q_int + extra))
    # Cap size
    thr = thr[:MAX_PREDS_PER_COL]
    # MSRP sanity cap
    if colname == "MSRP":
        thr = [t for t in thr if t >= MSRP_MIN_CAP]
    return thr

# ----------------------------
# Build predicates in U (bitsets over U rows)
# ----------------------------
def build_predicates_in_U(U: pd.DataFrame):
    preds = []
    nU = len(U)

    # Categorical
    for col in CAT_EXTRA_COLS:
        if col not in U.columns:
            continue
        vc = U[col].value_counts()
        values = vc[vc >= MIN_CAT_SUPPORT_IN_U].index.tolist()[:MAX_PREDS_PER_COL]
        for v in values:
            maskU = (U[col].values == v)
            if maskU.all() or (~maskU).all():
                continue
            preds.append({"col": col, "op": "==", "value": v, "bits": make_bitset_from_bool(maskU)})

    # Numeric
    for col, op in NUM_EXTRA_SPECS:
        if col not in U.columns:
            continue
        thr = numeric_thresholds(U[col], col, op)
        for t in thr:
            if op == ">=":
                maskU = (U[col].values >= t)
            else:
                maskU = (U[col].values <= t)
            if maskU.all() or (~maskU).all():
                continue
            preds.append({"col": col, "op": op, "value": int(t), "bits": make_bitset_from_bool(maskU)})

    # Dedup (col,op,value)
    uniq = {}
    for p in preds:
        uniq[(p["col"], p["op"], p["value"])] = p
    preds = list(uniq.values())

    # Prune too weak / too strong
    filtered = []
    for p in preds:
        cnt = popcount(p["bits"])
        if cnt < 2 or cnt > int(0.92 * nU):
            continue
        filtered.append(p)
    preds = filtered

    # Prefer informative predicates (support near middle), but keep enough diversity
    by_col = {}
    for p in preds:
        by_col.setdefault(p["col"], []).append(p)
    for col in by_col:
        by_col[col].sort(key=lambda p: abs(popcount(p["bits"]) - nU/2))

    capped = []
    for col, plist in by_col.items():
        capped.extend(plist[:MAX_PREDS_PER_COL])

    capped.sort(key=lambda p: abs(popcount(p["bits"]) - nU/2))
    return capped[:MAX_TOTAL_PREDS]

# ----------------------------
# Find MUS-4: U∩abcd empty but any 3-way non-empty
# Guided approach: enumerate "good triples" then find 4th
# ----------------------------
def find_drop_any_one_quad(preds, require_distinct_cols=True, max_good_triples=25000):
    n = len(preds)
    if n < 4:
        return None

    # Precompute good triples: intersection non-empty
    triples = []
    for i, j, k in combinations(range(n), 3):
        if require_distinct_cols and len({preds[i]["col"], preds[j]["col"], preds[k]["col"]}) < 3:
            continue
        bits_ijk = preds[i]["bits"] & preds[j]["bits"] & preds[k]["bits"]
        if bits_ijk != 0:
            triples.append((i, j, k, bits_ijk))
            if len(triples) >= max_good_triples:
                break

    if not triples:
        return None

    random.shuffle(triples)

    # For each triple, find l s.t. full empty + other 3-way intersections non-empty
    for i, j, k, bits_ijk in triples:
        for l in range(n):
            if l in (i, j, k):
                continue
            if require_distinct_cols and len({preds[i]["col"], preds[j]["col"], preds[k]["col"], preds[l]["col"]}) < 4:
                continue

            if (bits_ijk & preds[l]["bits"]) != 0:
                continue  # full not empty

            if (preds[i]["bits"] & preds[j]["bits"] & preds[l]["bits"]) == 0:
                continue
            if (preds[i]["bits"] & preds[k]["bits"] & preds[l]["bits"]) == 0:
                continue
            if (preds[j]["bits"] & preds[k]["bits"] & preds[l]["bits"]) == 0:
                continue

            return [preds[i], preds[j], preds[k], preds[l]]

    return None

# ----------------------------
# Natural language rendering
# ----------------------------
def clause(col, op, value):
    if op == "==":
        if col == "Engine Fuel Type":
            return f"runs on {value}"
        if col == "Vehicle Size":
            return f"is {str(value).lower()} size"
        if col == "Model":
            return f"is a {value}"
        return f"has {col} {value}"
    v = int(value)
    if col == "Year" and op == ">=":
        return f"is from {v} or newer"
    if col == "city mpg" and op == ">=":
        return f"gets at least {v} city MPG"
    if col == "highway MPG" and op == ">=":
        return f"gets at least {v} highway MPG"
    if col == "MSRP" and op == "<=":
        return f"has MSRP at most ${v:,}"
    return f"has {col} {op} {v}"

def make_base_query_sentence(base_rec):
    return (
        f"I am looking for a {base_rec['Make']} {base_rec['Vehicle Style']} "
        f"with a {base_rec['Transmission Type']} transmission and {base_rec['Driven_Wheels']}."
    )

def make_full_query_sentence(base_rec, additional):
    base_part = (
        f"I am looking for a {base_rec['Make']} {base_rec['Vehicle Style']} "
        f"with a {base_rec['Transmission Type']} transmission and {base_rec['Driven_Wheels']}"
    )
    extras = ", ".join(clause(p["col"], p["op"], p["value"]) for p in additional)
    return base_part + ", " + extras + "."

# ----------------------------
# Weights + oracle recommendation (unchanged)
# ----------------------------
def sample_weights(additional, rng: np.random.Generator):
    raw = rng.uniform(0.2, 1.0, size=len(additional))
    w = raw / raw.sum()
    return [{"column": p["col"], "op": p["op"], "value": p["value"], "weight": float(wi)} for p, wi in zip(additional, w)]

def apply_constraint_to_df(df_in: pd.DataFrame, p: dict) -> np.ndarray:
    col, op, value = p["col"], p["op"], p["value"]
    if op == "==":
        return (df_in[col].values == value)
    if op == ">=":
        return (df_in[col].values >= int(value))
    if op == "<=":
        return (df_in[col].values <= int(value))
    raise ValueError(f"Unsupported op: {op}")

def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x)); xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def recommend_car_for_instance(U: pd.DataFrame, additional: list, weighted_constraints: list):
    w_sorted = sorted(weighted_constraints, key=lambda d: d["weight"])
    relaxed = w_sorted[0]

    keep = [p for p in additional if not (p["col"] == relaxed["column"] and p["op"] == relaxed["op"] and p["value"] == relaxed["value"])]

    mask = np.ones(len(U), dtype=bool)
    for p in keep:
        mask &= apply_constraint_to_df(U, p)

    candidates = U[mask].copy()
    if len(candidates) == 0:
        return {"relaxed_constraint": relaxed, "recommended_car": None}

    scores = np.zeros(len(candidates), dtype=float)
    norms = {}
    for col in ["Year", "city mpg", "highway MPG", "MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)
    cand_pos_in_U = candidates.index.values

    for wc in weighted_constraints:
        col, op, val, w = wc["column"], wc["op"], wc["value"], wc["weight"]
        if op == "==":
            scores += w * (candidates[col].values == val).astype(float)
        elif op == ">=":
            scores += w * (norms[col][cand_pos_in_U] if col in norms else (candidates[col].values >= int(val)).astype(float))
        elif op == "<=":
            scores += w * ((1.0 - norms[col][cand_pos_in_U]) if col in norms else (candidates[col].values <= int(val)).astype(float))

    msrp = candidates["MSRP"].values.astype(float) if "MSRP" in candidates.columns else np.zeros(len(candidates), dtype=float)
    best_idx = np.lexsort((msrp, -scores))[0]
    best_row = candidates.iloc[best_idx]

    recommended = {
        "Make": best_row.get("Make", None),
        "Model": best_row.get("Model", None),
        "Year": int(best_row["Year"]) if "Year" in best_row else None,
        "Engine Fuel Type": best_row.get("Engine Fuel Type", None),
        "Transmission Type": best_row.get("Transmission Type", None),
        "Driven_Wheels": best_row.get("Driven_Wheels", None),
        "Vehicle Size": best_row.get("Vehicle Size", None),
        "Vehicle Style": best_row.get("Vehicle Style", None),
        "city mpg": int(best_row["city mpg"]) if "city mpg" in best_row else None,
        "highway MPG": int(best_row["highway MPG"]) if "highway MPG" in best_row else None,
        "MSRP": int(best_row["MSRP"]) if "MSRP" in best_row else None,
    }
    return {"relaxed_constraint": relaxed, "recommended_car": recommended}

# ----------------------------
# Generate TARGET_QUERIES
# ----------------------------
results = []
seen = set()

tries = 0
for base_rec in eligible_bases:
    tries += 1
    if tries > MAX_BASE_TRIES or len(results) >= TARGET_QUERIES:
        break

    base_sig = tuple(base_rec[c] for c in BASE_COLS)
    if base_sig in seen:
        continue

    U_idx = get_base_indices(base_rec)
    if not (BASE_MIN <= len(U_idx) <= BASE_MAX):
        continue

    U = df.iloc[U_idx].reset_index(drop=True)
    preds = build_predicates_in_U(U)

    additional = find_drop_any_one_quad(preds, require_distinct_cols=True)
    if additional is None:
        continue

    # Quality scoring: prefer tighter instances (drop-any-one intersections not huge)
    # This avoids widening BASE_MAX harming "tightness"
    # Compute drop-any-one counts within U
    bits = [p["bits"] for p in additional]
    drop_counts = []
    ok = True
    for i in range(4):
        inter = bits[(i+1)%4] & bits[(i+2)%4] & bits[(i+3)%4]
        c = popcount(inter)
        if c == 0:
            ok = False
            break
        drop_counts.append(c)
    if not ok:
        continue
    tight_score = sum(drop_counts)  # lower = tighter
    if tight_score > 180:  # reject very loose cases
        continue

    seen.add(base_sig)

    base_query_sentence = make_base_query_sentence(base_rec)
    full_query_sentence = make_full_query_sentence(base_rec, additional)

    additional_constraints = [{"column": p["col"], "op": p["op"], "value": p["value"]} for p in additional]
    weighted_constraints = sample_weights(additional, rng)

    rec = recommend_car_for_instance(U=U, additional=additional, weighted_constraints=weighted_constraints)

    results.append({
        "base_query_sentence": base_query_sentence,
        "full_query_sentence": full_query_sentence,
        "additional_constraints": additional_constraints,
        "constraint_weights": weighted_constraints,
        "relaxation_policy": "relax_lowest_weight_constraint",
        "chosen_relaxation": rec["relaxed_constraint"],
        "recommended_car": rec["recommended_car"],
        "meta": {"U_size": int(len(U)), "tight_score": int(tight_score), "drop_counts": drop_counts}
    })

print(f"Generated {len(results)} queries after trying {tries} base groups.")

if len(results) < TARGET_QUERIES:
    print(
        f"\nWARNING: Could not reach {TARGET_QUERIES}. "
        f"Got {len(results)}. Next best quality-preserving steps:\n"
        f"  1) increase MAX_BASE_TRIES (search more bases)\n"
        f"  2) slightly raise BASE_MAX (e.g., 450) but keep tight_score filter\n"
        f"  3) increase MAX_TOTAL_PREDS modestly (e.g., 180) for more predicate variety\n"
        f"  4) if still stuck, lower require_distinct_cols ONLY as last resort\n"
    )

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nWrote {len(results)} queries to: {OUT_PATH}")
if results:
    print("Example record:\n", json.dumps(results[0], indent=2, ensure_ascii=False))
